In [1]:
import numpy as np
import stim
import CircuitBuilder
import pymatching as pm
import statevector as sv

In [27]:

# Make a circuit builder with state setup
def prepare_circuit_builder(nancillae, er):
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    if nancillae == 2:
        cb.al('R 0 1 2 3 4 5')
    elif nancillae == 4:
        cb.al('R 0 1 2 3 4 5 6 7')
    cb.al('H 0')
    cb.al('CX 0 1')
    cb.al('CX 0 2')
    cb.al('CX 0 3')
    return cb

In [ ]:
# 2 stabilizer checks per round
def standard422(nrounds, er):
    cb = prepare_circuit_builder(2, er)
    # ZZZZ check on ancilla 1 (qubit 4)
    # XXXX check on ancilla 2 (qubit 5)
    # Initial measurements on ancillas for decoders
    cb.al(f'MR(0) 4 5')
    cb.al('H 5')

    cb.begin_loop(nrounds)

    # Time window 1
    cb.ale('CX 0 4')
    cb.ale('CX 5 1')
    cb.time_tick()
    # Time window 2
    cb.ale('CX 1 4')
    cb.ale('CX 5 2')
    cb.time_tick()
    # Time window 3
    cb.ale('CX 2 4')
    cb.ale('CX 5 3')
    cb.time_tick()
    # Time window 4
    cb.ale('CX 3 4')
    cb.ale('CX 5 0')
    cb.time_tick()

    # Measurement
    cb.ale('H 5')
    cb.ale(f'MR({er}) 4 5')
    cb.ale('H 5')
    cb.add_detector([-1, -3])
    cb.add_detector([-2, -4])
    cb.time_tick()

    cb.end_loop()

    # Final measurement
    cb.ale(f'MR({er}) 0 1')
    cb.add_detector([-1, -2])
    return cb.get_circuit()

def gauge422(nrounds, er):
    cb = prepare_circuit_builder(4, er)
    # ZZII anc 1, IIZZ anc 2, XX
    cb.al(f'MR(0) 4 5 6 7')
    cb.al

In [110]:
cb = standard422(1, 0.001)
cb.diagram()

/------------------------------------------------------------\ /----------------------------------------------------------\ /----------------------------------------------------------\ /----------------------------------------------------------\ /--------------------------------------------------------------------------------------------------------------------\ /--------------------------------------------------------------------------\
q0: -R-H-@-@-@------------@-DEPOLARIZE2(0.001)----------------------DEPOLARIZE1(0.001)-------------------------------------------DEPOLARIZE1(0.001)-------------------------------------------DEPOLARIZE1(0.001)----------------------X-DEPOLARIZE2(0.001)-DEPOLARIZE1(0.001)-----------------------------------------------------------------------------------------------------DEPOLARIZE1(0.001)-MR(0.001):rec[4]-X_ERROR(0.001)-DETECTOR:D2=rec[5]*rec[4]-
         | | |            | |                                                                                                                                                                                                         | |
q1: -R---X-|-|------------|-|------------------X-DEPOLARIZE2(0.001)-DEPOLARIZE1(0.001)-@-DEPOLARIZE2(0.001)----------------------DEPOLARIZE1(0.001)-------------------------------------------DEPOLARIZE1(0.001)----------------------|-|------------------DEPOLARIZE1(0.001)-----------------------------------------------------------------------------------------------------DEPOLARIZE1(0.001)-MR(0.001):rec[5]-X_ERROR(0.001)---------------------------
           | |            | |                  | |                                     | |                                                                                                                                            | |
q2: -R-----X-|------------|-|------------------|-|------------------DEPOLARIZE1(0.001)-|-|------------------X-DEPOLARIZE2(0.001)-DEPOLARIZE1(0.001)-@-DEPOLARIZE2(0.001)----------------------DEPOLARIZE1(0.001)----------------------|-|------------------DEPOLARIZE1(0.001)-----------------------------------------------------------------------------------------------------DEPOLARIZE1(0.001)-----------------------------------------------------------
             |            | |                  | |                                     | |                  | |                                     | |                                                                               | |
q3: -R-------X------------|-|------------------|-|------------------DEPOLARIZE1(0.001)-|-|------------------|-|------------------DEPOLARIZE1(0.001)-|-|------------------X-DEPOLARIZE2(0.001)-DEPOLARIZE1(0.001)-@-DEPOLARIZE2(0.001)-|-|------------------DEPOLARIZE1(0.001)-----------------------------------------------------------------------------------------------------DEPOLARIZE1(0.001)-----------------------------------------------------------
                          | |                  | |                                     | |                  | |                                     | |                  | |                                     | |                  | |
q4: -R-------MR(0):rec[0]-X-DEPOLARIZE2(0.001)-|-|------------------DEPOLARIZE1(0.001)-X-DEPOLARIZE2(0.001)-|-|------------------DEPOLARIZE1(0.001)-X-DEPOLARIZE2(0.001)-|-|------------------DEPOLARIZE1(0.001)-X-DEPOLARIZE2(0.001)-|-|------------------DEPOLARIZE1(0.001)---MR(0.001):rec[2]---X_ERROR(0.001)---------------------------------------DETECTOR:D1=rec[2]*rec[0]-DEPOLARIZE1(0.001)-----------------------------------------------------------
                                               | |                                                          | |                                                          | |                                                          | |
q5: -R-------MR(0):rec[1]-H--------------------@-DEPOLARIZE2(0.001)-DEPOLARIZE1(0.001)----------------------@-DEPOLARIZE2(0.001)-DEPOLARIZE1(0.001)-

In [158]:
nrounds = 5
er = 0.001
c = standard422(nrounds, er)
sample = c.compile_detector_sampler().sample(100000)
print(len(sample), np.sum(sample[:,nrounds*2]))
filtered = sample
for i in range(nrounds*2):
    filtered = filtered[filtered[:,i] == 0]
print(len(filtered), np.sum(filtered[:,nrounds*2]))


100000 5812
110 6
